<div style="background: linear-gradient(135deg, #0f766e 0%, #155e75 100%); color: white; padding: 24px 28px; border-radius: 14px; margin-bottom: 18px;">
  <h1 style="margin: 0; font-size: 30px;">Urban Mobility Analytics</h1>
  <p style="margin: 8px 0 0 0; font-size: 17px;">Azure, Microsoft Fabric, Lakehouse, PySpark, Dataflow Gen2, and Power BI</p>
</div>

## Project Overview

**Goal:** Build an end-to-end analytics pipeline using Azure Blob Storage, Microsoft Fabric Pipeline, Lakehouse, Notebook, Dataflow Gen2, and Power BI. The project ingests NYC taxi trip data, transforms it into analytics-ready tables, and powers a dashboard for business insights.

**Dataset source:** This project uses the **NYC Taxi & Limousine Commission yellow taxi trip records** published through **Azure Open Datasets**. Microsoft describes this dataset as including pickup and dropoff dates and times, pickup and dropoff locations, trip distances, itemized fares, rate types, payment types, and driver-reported passenger counts.

**Project scale:** The current project slice covers **6 months of 2019 data** and processes **more than 52 million trip records** across the Bronze, Silver, and Gold pipeline. This makes the project materially larger than a classroom-sized sample and much closer to a realistic analytics workload.

**Why the dataset is so large:** Each row represents a single taxi trip, and New York City generates a very high volume of trips every day. When that trip-level detail is retained across multiple months, the row count grows rapidly into tens of millions. Each trip also includes multiple analytical fields such as timestamps, distance, fare, payment type, passenger count, and location identifiers, which increases both storage volume and processing complexity.

**Why this scale matters for analysis and reporting:** A large trip-level dataset improves the reliability of trend analysis, reduces the impact of small-sample noise, supports richer daily and weekly time-series reporting, and creates a stronger foundation for forecasting, operational analysis, and executive reporting. For a portfolio project, this scale is significant because it demonstrates the ability to work with realistic data engineering volumes rather than a tiny demonstration extract.


<div style="background:#ecfdf5; border: 1px solid #bbf7d0; border-left: 6px solid #16a34a; padding: 14px 18px; border-radius: 10px; margin: 14px 0;">
  <h2 style="margin: 0; color:#166534;">Architecture and Medallion Design</h2>
  <p style="margin: 8px 0 0 0; color:#052e16;"><strong>Pipeline architecture:</strong></p>
  <ol style="margin: 10px 0 0 20px; color:#052e16;">
    <li>Azure Blob Storage stores the raw taxi parquet files.</li>
    <li>Fabric Data Pipeline copies the raw files into the Lakehouse.</li>
    <li>Fabric Notebook cleans and enriches the trip data.</li>
    <li>Dataflow Gen2 shapes the cleaned data for reporting.</li>
    <li>Power BI visualizes KPIs, trends, and time-series insights.</li>
  </ol>
  <p style="margin: 12px 0 0 0; color:#052e16;"><strong>Bronze / Silver / Gold design:</strong></p>
  <ul style="margin: 10px 0 0 20px; color:#052e16;">
    <li><strong>Bronze</strong> stores the raw source data copied from Azure Blob Storage into the Lakehouse.</li>
    <li><strong>Silver</strong> stores cleaned and standardized trip-level data. In this notebook, <code>silver_trips_clean</code> is created after data type fixes, quality filters, and derived analytics columns.</li>
    <li><strong>Gold</strong> stores business-ready reporting data. In this project, <code>gold_trips_report</code> is built from the silver layer for Power BI reporting.</li>
  </ul>
  <p style="margin: 10px 0 0 0; color:#052e16;">This separation makes the project easier to explain, debug, scale, and present in GitHub because each layer has a clear role in the analytics workflow.</p>
</div>


# Silver Layer Design: Building `silver_trips_clean` in the Notebook

This notebook represents the **Silver layer** of the project. By the time this stage begins, the **Bronze layer** has already been completed: raw taxi parquet files have been stored in Azure Blob Storage and ingested into the project Lakehouse through a Fabric Data Pipeline.

## Notebook Purpose

- Read raw NYC taxi data from the Lakehouse.
- Identify schema patterns and key business columns.
- Standardize column names and data types.
- Apply data quality rules.
- Create derived analytical features.
- Write the curated Silver table: `silver_trips_clean`.

## Expected Outputs

- `silver_trips_clean`
- Row-count validation checks
- Summary metrics for quality review


<div style="background:#e0f2fe; border-left: 6px solid #0284c7; padding: 14px 18px; border-radius: 10px; margin: 14px 0;">
  <h2 style="margin: 0; color:#075985;">Step 01: Set Up Environment and Load Raw Data</h2>
  <p style="margin: 8px 0 0 0; color:#0f172a;">This step imports the PySpark functions used throughout the notebook, defines the source and target table names, reads the raw Lakehouse table, and previews the first records. The preview confirms that the ingestion pipeline successfully made the NYC taxi data available for transformation.</p>
</div>


In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

source_table_name = "nyc-taxi-raw"
silver_table_name = "silver_trips_clean"
raw_df = spark.sql(f"SELECT * FROM `{source_table_name}`")

#Read the raw parquet files from the Lakehouse
#This code reads all parquet files from the Lakehouse Files area. Since your source files were selected as parquet part-... files,
#Spark can read them directly.

raw_row_count = raw_df.count()
print(f"Raw row count: {raw_row_count:,}")

display(raw_df.limit(10))



StatementMeta(, e9cb709b-e57e-4ab1-b997-7bd76e962472, 3, Finished, Available, Finished, False)

Raw row count: 52,120,845


SynapseWidget(Synapse.DataFrame, 17da46c5-fb38-46db-b4f7-3fc84dad5e50)

<div style="background:#fafaf9; padding: 10px 14px; border-radius: 8px; margin: 10px 0 18px 0;">
  <p style="margin: 0; font-size: 12px; letter-spacing: 0.04em; text-transform: uppercase; color:#78716c;">Interpretation</p>
  <p style="margin: 6px 0 0 0; color:#44403c;">This preview confirms that the raw NYC taxi trip data has been loaded successfully from the Lakehouse table. Each row represents one taxi trip and includes fields such as pickup and dropoff timestamps, passenger count, trip distance, fare, payment type, and location information. At this stage, the data is still considered raw because the schema, data types, and data quality rules have not yet been standardized.</p>
</div>


<div style="background:#e0f2fe; border-left: 6px solid #0284c7; padding: 14px 18px; border-radius: 10px; margin: 14px 0;">
  <h2 style="margin: 0; color:#075985;">Step 02: Inspect Schema and Incoming Columns</h2>
  <p style="margin: 8px 0 0 0; color:#0f172a;">This step inspects the schema before cleaning. It confirms the available fields, data types, and naming pattern so the transformation logic can map the correct pickup, dropoff, fare, distance, and payment columns.</p>
</div>


In [2]:
raw_df.printSchema()

print("Columns:")
for column_name in raw_df.columns:
    print(f"- {column_name}")

StatementMeta(, e9cb709b-e57e-4ab1-b997-7bd76e962472, 4, Finished, Available, Finished, False)

root
 |-- vendorID: string (nullable = true)
 |-- tpepPickupDateTime: timestamp (nullable = true)
 |-- tpepDropoffDateTime: timestamp (nullable = true)
 |-- passengerCount: integer (nullable = true)
 |-- tripDistance: double (nullable = true)
 |-- puLocationId: string (nullable = true)
 |-- doLocationId: string (nullable = true)
 |-- startLon: double (nullable = true)
 |-- startLat: double (nullable = true)
 |-- endLon: double (nullable = true)
 |-- endLat: double (nullable = true)
 |-- rateCodeId: integer (nullable = true)
 |-- storeAndFwdFlag: string (nullable = true)
 |-- paymentType: string (nullable = true)
 |-- fareAmount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mtaTax: double (nullable = true)
 |-- improvementSurcharge: string (nullable = true)
 |-- tipAmount: double (nullable = true)
 |-- tollsAmount: double (nullable = true)
 |-- totalAmount: double (nullable = true)

Columns:
- vendorID
- tpepPickupDateTime
- tpepDropoffDateTime
- passengerCount
- t

<div style="background:#e0f2fe; border-left: 6px solid #0284c7; padding: 14px 18px; border-radius: 10px; margin: 14px 0;">
  <h2 style="margin: 0; color:#075985;">Step 03: Normalize Column Names</h2>
  <p style="margin: 8px 0 0 0; color:#0f172a;">This step standardizes all column names to lowercase. Normalized names reduce errors caused by inconsistent capitalization and make later column matching easier.</p>
</div>


In [3]:
# Standardize all column names to lowercase
df = raw_df
for old_name in raw_df.columns:
    df = df.withColumnRenamed(old_name, old_name.lower())

print("Normalized columns:")
print(df.columns)

StatementMeta(, e9cb709b-e57e-4ab1-b997-7bd76e962472, 5, Finished, Available, Finished, False)

Normalized columns:
['vendorid', 'tpeppickupdatetime', 'tpepdropoffdatetime', 'passengercount', 'tripdistance', 'pulocationid', 'dolocationid', 'startlon', 'startlat', 'endlon', 'endlat', 'ratecodeid', 'storeandfwdflag', 'paymenttype', 'fareamount', 'extra', 'mtatax', 'improvementsurcharge', 'tipamount', 'tollsamount', 'totalamount']


<div style="background:#e0f2fe; border-left: 6px solid #0284c7; padding: 14px 18px; border-radius: 10px; margin: 14px 0;">
  <h2 style="margin: 0; color:#075985;">Step 04: Resolve Key Business Columns</h2>
  <p style="margin: 8px 0 0 0; color:#0f172a;">This step maps the dataset columns to business concepts such as pickup time, dropoff time, trip distance, fare, total amount, tip, passenger count, and payment type. The mapping supports multiple naming styles, including snake_case and Fabric-style camelCase converted to lowercase.</p>
</div>


In [4]:
def first_existing(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None

cols = df.columns

pickup_col = first_existing(cols, [
    "tpep_pickup_datetime",
    "tpeppickupdatetime",
    "lpep_pickup_datetime",
    "lpeppickupdatetime",
    "pickup_datetime",
    "pickupdatetime"
])

dropoff_col = first_existing(cols, [
    "tpep_dropoff_datetime",
    "tpepdropoffdatetime",
    "lpep_dropoff_datetime",
    "lpepdropoffdatetime",
    "dropoff_datetime",
    "dropoffdatetime"
])

distance_col = first_existing(cols, [
    "trip_distance",
    "tripdistance"
])

fare_col = first_existing(cols, [
    "fare_amount",
    "fareamount"
])

total_col = first_existing(cols, [
    "total_amount",
    "totalamount"
])

tip_col = first_existing(cols, [
    "tip_amount",
    "tipamount"
])

passenger_col = first_existing(cols, [
    "passenger_count",
    "passengercount"
])

payment_col = first_existing(cols, [
    "payment_type",
    "paymenttype"
])

resolved_columns = {
    "pickup_col": pickup_col,
    "dropoff_col": dropoff_col,
    "distance_col": distance_col,
    "fare_col": fare_col,
    "total_col": total_col,
    "tip_col": tip_col,
    "passenger_col": passenger_col,
    "payment_col": payment_col,
}

for label, value in resolved_columns.items():
    print(f"{label}: {value}")



StatementMeta(, e9cb709b-e57e-4ab1-b997-7bd76e962472, 6, Finished, Available, Finished, False)

pickup_col: tpeppickupdatetime
dropoff_col: tpepdropoffdatetime
distance_col: tripdistance
fare_col: fareamount
total_col: totalamount
tip_col: tipamount
passenger_col: passengercount
payment_col: paymenttype


<div style="background:#e0f2fe; border-left: 6px solid #0284c7; padding: 14px 18px; border-radius: 10px; margin: 14px 0;">
  <h2 style="margin: 0; color:#075985;">Step 05: Cast Important Fields</h2>
  <p style="margin: 8px 0 0 0; color:#0f172a;">This step converts important fields into analysis-ready data types. Timestamp fields are converted for time calculations, numeric fields are converted for aggregation, and passenger count is converted for filtering.</p>
</div>


In [5]:
typed_df = df

if pickup_col:
    typed_df = typed_df.withColumn(pickup_col, F.to_timestamp(F.col(pickup_col)))
if dropoff_col:
    typed_df = typed_df.withColumn(dropoff_col, F.to_timestamp(F.col(dropoff_col)))
if distance_col:
    typed_df = typed_df.withColumn(distance_col, F.col(distance_col).cast("double"))
if fare_col:
    typed_df = typed_df.withColumn(fare_col, F.col(fare_col).cast("double"))
if total_col:
    typed_df = typed_df.withColumn(total_col, F.col(total_col).cast("double"))
if tip_col:
    typed_df = typed_df.withColumn(tip_col, F.col(tip_col).cast("double"))
if passenger_col:
    typed_df = typed_df.withColumn(passenger_col, F.col(passenger_col).cast("int"))

typed_df.printSchema()
display(typed_df.limit(10))


StatementMeta(, e9cb709b-e57e-4ab1-b997-7bd76e962472, 7, Finished, Available, Finished, False)

root
 |-- vendorid: string (nullable = true)
 |-- tpeppickupdatetime: timestamp (nullable = true)
 |-- tpepdropoffdatetime: timestamp (nullable = true)
 |-- passengercount: integer (nullable = true)
 |-- tripdistance: double (nullable = true)
 |-- pulocationid: string (nullable = true)
 |-- dolocationid: string (nullable = true)
 |-- startlon: double (nullable = true)
 |-- startlat: double (nullable = true)
 |-- endlon: double (nullable = true)
 |-- endlat: double (nullable = true)
 |-- ratecodeid: integer (nullable = true)
 |-- storeandfwdflag: string (nullable = true)
 |-- paymenttype: string (nullable = true)
 |-- fareamount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mtatax: double (nullable = true)
 |-- improvementsurcharge: string (nullable = true)
 |-- tipamount: double (nullable = true)
 |-- tollsamount: double (nullable = true)
 |-- totalamount: double (nullable = true)



SynapseWidget(Synapse.DataFrame, 4dddd0be-5495-487b-a9bf-a2169c5c9d01)

<div style="background:#fafaf9; padding: 10px 14px; border-radius: 8px; margin: 10px 0 18px 0;">
  <p style="margin: 0; font-size: 12px; letter-spacing: 0.04em; text-transform: uppercase; color:#78716c;">Interpretation</p>
  <p style="margin: 6px 0 0 0; color:#44403c;">This result shows the dataset after important fields have been converted into analysis-ready data types. Timestamp columns are now suitable for time-based calculations, numeric fields such as fare and trip distance can be aggregated correctly, and passenger count can be filtered consistently. The row values remain similar to the raw preview, but the schema is now safer and more reliable for transformation and reporting.</p>
</div>


<div style="background:#e0f2fe; border-left: 6px solid #0284c7; padding: 14px 18px; border-radius: 10px; margin: 14px 0;">
  <h2 style="margin: 0; color:#075985;">Step 06: Apply Data Quality Filters</h2>
  <p style="margin: 8px 0 0 0; color:#0f172a;">This step removes records that would distort analysis, such as missing timestamps, non-positive trip distance, negative fare or total amounts, and unrealistic passenger counts. The result is a cleaner trip-level dataset.</p>
</div>


In [6]:
clean_df = typed_df

if pickup_col:
    clean_df = clean_df.filter(F.col(pickup_col).isNotNull())
if dropoff_col:
    clean_df = clean_df.filter(F.col(dropoff_col).isNotNull())
if distance_col:
    clean_df = clean_df.filter(F.col(distance_col) > 0)
if fare_col:
    clean_df = clean_df.filter(F.col(fare_col) >= 0)
if total_col:
    clean_df = clean_df.filter(F.col(total_col) >= 0)
if passenger_col:
    clean_df = clean_df.filter((F.col(passenger_col).isNull()) | ((F.col(passenger_col) >= 0) & (F.col(passenger_col) <= 8)))

print("Clean row count:", clean_df.count())
display(clean_df.limit(10))


StatementMeta(, e9cb709b-e57e-4ab1-b997-7bd76e962472, 8, Finished, Available, Finished, False)

Clean row count: 51676828


SynapseWidget(Synapse.DataFrame, 01f67a5c-c8cb-40e4-9ea0-00949112ba51)

<div style="background:#fafaf9; padding: 10px 14px; border-radius: 8px; margin: 10px 0 18px 0;">
  <p style="margin: 0; font-size: 12px; letter-spacing: 0.04em; text-transform: uppercase; color:#78716c;">Interpretation</p>
  <p style="margin: 6px 0 0 0; color:#44403c;">This preview shows the dataset after the main data quality filters have been applied. Records with missing timestamps, non-positive trip distance, negative fare or total values, and unrealistic passenger counts have been removed. The result is a more trustworthy trip-level dataset for dashboard metrics such as total trips, average fare, and revenue trends.</p>
</div>


<div style="background:#e0f2fe; border-left: 6px solid #0284c7; padding: 14px 18px; border-radius: 10px; margin: 14px 0;">
  <h2 style="margin: 0; color:#075985;">Step 07: Create Derived Analytics Columns</h2>
  <p style="margin: 8px 0 0 0; color:#0f172a;">This step creates dashboard-friendly fields such as pickup date, year, month, hour, weekday, trip duration, and revenue per mile. These fields support trend analysis, operational KPIs, and Power BI visuals.</p>
</div>


In [7]:
enriched_df = clean_df

if pickup_col:
    enriched_df = (
        enriched_df
        .withColumn("pickup_date", F.to_date(F.col(pickup_col)))
        .withColumn("pickup_year", F.year(F.col(pickup_col)))
        .withColumn("pickup_month", F.month(F.col(pickup_col)))
        .withColumn("pickup_hour", F.hour(F.col(pickup_col)))
        .withColumn("pickup_weekday_num", F.dayofweek(F.col(pickup_col)))
        .withColumn(
            "pickup_weekday",
            F.when(F.col("pickup_weekday_num") == 1, F.lit("Sunday"))
             .when(F.col("pickup_weekday_num") == 2, F.lit("Monday"))
             .when(F.col("pickup_weekday_num") == 3, F.lit("Tuesday"))
             .when(F.col("pickup_weekday_num") == 4, F.lit("Wednesday"))
             .when(F.col("pickup_weekday_num") == 5, F.lit("Thursday"))
             .when(F.col("pickup_weekday_num") == 6, F.lit("Friday"))
             .otherwise(F.lit("Saturday"))
        )
    )

if pickup_col and dropoff_col:
    enriched_df = enriched_df.withColumn(
        "trip_duration_minutes",
        (F.col(dropoff_col).cast("long") - F.col(pickup_col).cast("long")) / 60.0
    )

if total_col and distance_col:
    enriched_df = enriched_df.withColumn(
        "revenue_per_mile",
        F.when(F.col(distance_col) > 0, F.col(total_col) / F.col(distance_col)).otherwise(None)
    )

# Filter impossible trip durations if column exists
if "trip_duration_minutes" in enriched_df.columns:
    enriched_df = enriched_df.filter((F.col("trip_duration_minutes") > 0) & (F.col("trip_duration_minutes") <= 300))

display(enriched_df.limit(10))


StatementMeta(, e9cb709b-e57e-4ab1-b997-7bd76e962472, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 182eeefd-d665-4a1a-a31b-26202318a007)

<div style="background:#fafaf9; padding: 10px 14px; border-radius: 8px; margin: 10px 0 18px 0;">
  <p style="margin: 0; font-size: 12px; letter-spacing: 0.04em; text-transform: uppercase; color:#78716c;">Interpretation</p>
  <p style="margin: 6px 0 0 0; color:#44403c;">This result shows the cleaned data after analytical features have been added. Columns such as <code>pickup_date</code>, <code>pickup_year</code>, <code>pickup_month</code>, <code>pickup_hour</code>, <code>pickup_weekday</code>, <code>trip_duration_minutes</code>, and <code>revenue_per_mile</code> make the dataset far more useful for business analysis. These derived fields support Power BI visuals for demand patterns, hourly trends, weekday behavior, and trip efficiency.</p>
</div>


<div style="background:#e0f2fe; border-left: 6px solid #0284c7; padding: 14px 18px; border-radius: 10px; margin: 14px 0;">
  <h2 style="margin: 0; color:#075985;">Step 08: Select Silver Reporting Columns</h2>
  <p style="margin: 8px 0 0 0; color:#0f172a;">This step keeps the most useful fields for downstream reporting and removes unnecessary raw columns. The result is a focused silver-layer table that is easier to model in Dataflow Gen2 and Power BI.</p>
</div>


In [8]:
preferred_columns = [
    pickup_col,
    dropoff_col,
    passenger_col,
    payment_col,
    distance_col,
    fare_col,
    tip_col,
    total_col,
    "pickup_date",
    "pickup_year",
    "pickup_month",
    "pickup_hour",
    "pickup_weekday_num",
    "pickup_weekday",
    "trip_duration_minutes",
    "revenue_per_mile"
]

selected_columns = [c for c in preferred_columns if c and c in enriched_df.columns]

silver_df = enriched_df.select(*selected_columns)

print("Silver columns:")
print(silver_df.columns)
display(silver_df.limit(10))


StatementMeta(, e9cb709b-e57e-4ab1-b997-7bd76e962472, 10, Finished, Available, Finished, False)

Silver columns:
['tpeppickupdatetime', 'tpepdropoffdatetime', 'passengercount', 'paymenttype', 'tripdistance', 'fareamount', 'tipamount', 'totalamount', 'pickup_date', 'pickup_year', 'pickup_month', 'pickup_hour', 'pickup_weekday_num', 'pickup_weekday', 'trip_duration_minutes', 'revenue_per_mile']


SynapseWidget(Synapse.DataFrame, ad4d5cad-2b3c-491a-a62d-5513ccf61b89)

<div style="background:#fafaf9; padding: 10px 14px; border-radius: 8px; margin: 10px 0 18px 0;">
  <p style="margin: 0; font-size: 12px; letter-spacing: 0.04em; text-transform: uppercase; color:#78716c;">Interpretation</p>
  <p style="margin: 6px 0 0 0; color:#44403c;">This table is the curated silver dataset that will be used for downstream reporting. It keeps the most important trip, fare, time, and derived analytics fields while removing unnecessary raw columns. This makes the data easier to understand, model, and use in Dataflow Gen2 or Power BI.</p>
</div>


<div style="background:#e0f2fe; border-left: 6px solid #0284c7; padding: 14px 18px; border-radius: 10px; margin: 14px 0;">
  <h2 style="margin: 0; color:#075985;">Step 09: Run Quality Checks Before Saving</h2>
  <p style="margin: 8px 0 0 0; color:#0f172a;">This step validates the final silver dataset before writing it to the Lakehouse. Summary metrics such as revenue, average trip distance, and average trip duration help confirm that the transformation produced reasonable results.</p>
</div>


In [9]:
print("Final silver row count:", silver_df.count())

summary_exprs = []
if total_col and total_col in silver_df.columns:
    summary_exprs.append(F.sum(total_col).alias("total_revenue"))
    summary_exprs.append(F.avg(total_col).alias("avg_total_amount"))
if distance_col and distance_col in silver_df.columns:
    summary_exprs.append(F.avg(distance_col).alias("avg_trip_distance"))
if "trip_duration_minutes" in silver_df.columns:
    summary_exprs.append(F.avg("trip_duration_minutes").alias("avg_trip_duration_minutes"))

if summary_exprs:
    display(silver_df.agg(*summary_exprs))


StatementMeta(, e9cb709b-e57e-4ab1-b997-7bd76e962472, 11, Finished, Available, Finished, False)

Final silver row count: 51538969


SynapseWidget(Synapse.DataFrame, 70a00f13-6b6f-4df8-a3c7-3677c9e7d3c7)

<div style="background:#fafaf9; padding: 10px 14px; border-radius: 8px; margin: 10px 0 18px 0;">
  <p style="margin: 0; font-size: 12px; letter-spacing: 0.04em; text-transform: uppercase; color:#78716c;">Interpretation</p>
  <p style="margin: 6px 0 0 0; color:#44403c;">This summary validates the transformed dataset before it is written back to the Lakehouse. Metrics such as total revenue, average total amount, average trip distance, and average trip duration help confirm that the cleaned data is reasonable and suitable for downstream reporting. These figures are also useful to document in GitHub as evidence of data validation.</p>
</div>


<div style="background:#e0f2fe; border-left: 6px solid #0284c7; padding: 14px 18px; border-radius: 10px; margin: 14px 0;">
  <h2 style="margin: 0; color:#075985;">Step 10: Write the Curated Silver Table</h2>
  <p style="margin: 8px 0 0 0; color:#0f172a;">This step saves the cleaned and enriched DataFrame as the Delta table `silver_trips_clean` in the Lakehouse. The `silver` prefix shows that this table is cleaned, standardized, and still detailed at the trip level. Overwrite mode is used during development so the transformation can be rerun cleanly after improvements.</p>
</div>



In [10]:
silver_df.write.mode("overwrite").format("delta").saveAsTable(silver_table_name)

print(f"Saved table: {silver_table_name}")


StatementMeta(, e9cb709b-e57e-4ab1-b997-7bd76e962472, 12, Finished, Available, Finished, False)

Saved table: silver_trips_clean


<div style="background:#e0f2fe; border-left: 6px solid #0284c7; padding: 14px 18px; border-radius: 10px; margin: 14px 0;">
  <h2 style="margin: 0; color:#075985;">Step 11: Verify the Saved Table</h2>
  <p style="margin: 8px 0 0 0; color:#0f172a;">This step queries the saved table and previews the result. Successful output confirms that `silver_trips_clean` is available for Dataflow Gen2, semantic modeling, and Power BI reporting.</p>
</div>


In [11]:
saved_df = spark.sql(f"SELECT * FROM {silver_table_name}")
print("Saved table row count:", saved_df.count())
display(saved_df.limit(10))


StatementMeta(, e9cb709b-e57e-4ab1-b997-7bd76e962472, 13, Finished, Available, Finished, True)

Saved table row count: 51538969


SynapseWidget(Synapse.DataFrame, b507ab2a-6316-4b34-ac98-eb7ece76234e)

<div style="background:#fafaf9; padding: 10px 14px; border-radius: 8px; margin: 10px 0 18px 0;">
  <p style="margin: 0; font-size: 12px; letter-spacing: 0.04em; text-transform: uppercase; color:#78716c;">Interpretation</p>
  <p style="margin: 6px 0 0 0; color:#44403c;">This final preview confirms that <code>silver_trips_clean</code> was written successfully and can be queried as a Lakehouse table. At this point, the Silver transformation stage is complete, and the table is ready for Dataflow Gen2, semantic modeling, and Power BI reporting.</p>
</div>
